# Age and gender detection

This notebook detects faces with OpenCV's DNN face detector, a ResNet-10 SSD. It then estimates
each face's gender and age bucket with the pretrained Caffe models from Levi & Hassner (2015),
which were trained on the Adience dataset.

It runs **locally** (Jupyter, using a photo file or your webcam) or in **Google Colab**, where it
captures a photo through the browser webcam. The reusable code is in the `agegender` package in
this repository.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Fetch the code and the pretrained models (~94 MB) into the Colab VM.
    if not os.path.exists("Age-Gender-Detection"):
        !git clone -q https://github.com/SAMARTHJAISWAL/Age-Gender-Detection.git
    %cd Age-Gender-Detection
    !pip -q install gdown
    !python scripts/download_models.py

In [ ]:
import cv2
import matplotlib.pyplot as plt

from agegender.detector import AgeGenderDetector

detector = AgeGenderDetector()   # model files from $AGEGENDER_MODEL_DIR or ./models
print("Models loaded")

## Get an image

- **Colab:** click *Capture* to take a webcam photo.
- **Local:** set `IMAGE_PATH` (or the `AGEGENDER_IMAGE` environment variable) to a photo, or set
  `USE_LOCAL_WEBCAM = True` to grab one frame from the webcam.

In [ ]:
IMAGE_PATH = os.environ.get("AGEGENDER_IMAGE", "photo.jpg")
USE_LOCAL_WEBCAM = False

def take_photo_colab(quality=0.8):
    from base64 import b64decode
    import numpy as np
    from IPython.display import Javascript, display
    from google.colab.output import eval_js
    display(Javascript('''
      async function takePhoto(quality) {
        const div = document.createElement('div');
        const capture = document.createElement('button');
        capture.textContent = 'Capture';
        div.appendChild(capture);
        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => capture.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', quality);
      }'''))
    data = eval_js(f"takePhoto({quality})")
    buf = np.frombuffer(b64decode(data.split(",")[1]), dtype=np.uint8)
    return cv2.imdecode(buf, cv2.IMREAD_COLOR)

def grab_local_webcam(index=0):
    cap = cv2.VideoCapture(index)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError("Could not read from the webcam")
    return frame

if IN_COLAB:
    frame = take_photo_colab()
elif USE_LOCAL_WEBCAM:
    frame = grab_local_webcam()
else:
    frame = cv2.imread(IMAGE_PATH)
    if frame is None:
        raise FileNotFoundError(f"Set IMAGE_PATH to an existing photo (tried {IMAGE_PATH!r})")
print("Image size:", frame.shape[1], "x", frame.shape[0])

## Predict

In [ ]:
predictions = detector.predict(frame)
if not predictions:
    print("No face detected")
for i, p in enumerate(predictions, 1):
    print(f"Face {i}: {p.gender} ({p.gender_confidence:.2f}), age {p.age} ({p.age_confidence:.2f})")

annotated = detector.annotate(frame, predictions)
plt.figure(figsize=(8, 6)); plt.axis("off"); plt.imshow(annotated[:, :, ::-1]); plt.show()

## Notes

- The age model predicts one of 8 buckets: (0-2), (4-6), (8-12), (15-20), (25-32), (38-43),
  (48-53) and (60-100). It does not predict an exact age.
- These 2015-era models are known to be less accurate for some demographics and under poor
  lighting. Treat the output as a rough estimate, not as identity or age verification.